# Perception Test

In [1]:
# ASA Imports
from asa.affect_model.belief import AffectModel  # noqa: I001
from asa.core.observers import Observers
from asa.core.representations import EKMAN6
from asa.perception.decode_keyword import EKMAN6_KEYWORDS, KeywordDecoder
from asa.perception.text_console import TextConsole
from asa.runtime import run_agent


# Notebook Specifics / Temporary Functions
#

import logging
from asa._tools.custom_logging import setup_logging
setup_logging(level="DEBUG")
log = logging.getLogger("asa.observer.debug")


from asa.core.observers import Event  # noqa: E402, I001
from asa.core.affect import AffectEvidence, AffectState, Utterance  # noqa: E402

def read_or_end(prompt: str) -> str:
    """Notebook stand-in for Ctrl-D — no frontend here can send a real EOF."""
    text = input(prompt)
    if text.strip() == ":q":
        raise EOFError
    return text

# def log_event(event: Event) -> None:
#     """Temporary stand-in for the recorder — narrows by type so the fields are checked."""
#     when = event.at.strftime("%H:%M:%S.%f")[:-3]

#     if isinstance(event, Utterance):
#         # log.debug("heard    %s  %r", when, event.text)
#         log.debug("heard    %s  %s  %r", when, event.id, event.text)
#     elif isinstance(event, AffectEvidence):
#         fired = {k: round(v, 2) for k, v in event.affect.values.items() if v}
#         log.debug("evidence %s  %-5s %-16s %s", when, event.target, event.source, fired)
#     elif isinstance(event, AffectState):
#         log.debug("state    %s  other=%s self=%s", when,
#                   dict(event.other.values), dict(event.self_.values))
#     else:
#         log.debug("event    %s  %s  %r", when, event.schema, event)

#     # Full dataclass
#     log.debug("%-12s %s", event.schema, event)

def log_event(event: Event) -> None:
    """Temporary stand-in for the recorder — schema, time, id, then the payload."""
    when = event.at.strftime("%H:%M:%S.%f")[:-3]

    if isinstance(event, Utterance):
        log.debug("%-12s %s  %s  %r", event.schema, when, event.id, event.text)
    elif isinstance(event, AffectEvidence):
        fired = {str(k): round(v, 2) for k, v in event.affect.values.items() if v}
        log.debug("%-12s %s  %s  %-5s %-16s %s",
                  event.schema, when, event.of_input, event.target, event.source, fired)
    elif isinstance(event, AffectState):
        log.debug("%-12s %s  other=%s self=%s",
                  event.schema, when, dict(event.other.values), dict(event.self_.values))
    else:
        log.debug("%-12s %s  %r", event.schema, when, event)

    # Full dataclass
    log.debug("%-12s %s", event.schema, event)
    


# Text Console

In [ ]:
# Establish the source, decoder and a simple EKMAN6 keyword decoder
# source = TextConsole()
source = TextConsole(read=read_or_end)
decoder = KeywordDecoder(representation=EKMAN6, table=EKMAN6_KEYWORDS)

# Affectmodel is not implemented and just reports a stub
model = AffectModel()

# Observers is not implemented so just produce a debug log
observers = Observers()
observers.register(log_event)

# Kick-off the agent pipeline
await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)

## Test Text

In [2]:
# Get the root path and data paths
#

from pathlib import Path


def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")

DATA_IN = repo_root() / "data_in"



In [3]:
# Get the simple examples and BRIGHTER dataset into dfs
#

import pandas as pd

simple_df = pd.read_csv(DATA_IN / "simple_text_ekman6.csv")
simple_df["emotion"] = simple_df["emotion"].fillna("none").astype("category")

brighter_df = pd.read_parquet(DATA_IN / "brighter_emotions_raw.parquet")
brighter_df = brighter_df.rename(columns={"joy": "happiness"}, errors="raise")

In [4]:
from asa.core.affect import AffectVector, Utterance
from asa.core.representations import EKMAN6, AffectRepresentation

# Helper to create an AffectVector
#

def gen_affect_vector(rep: AffectRepresentation, **magnitudes: float) -> AffectVector:
    values = dict.fromkeys(rep.axes, rep.rest)
    for axis, magnitude in magnitudes.items():
        if axis not in values:
            raise ValueError(f"{axis!r} is not an axis of {rep.id}: {rep.axes}")
        values[axis] = magnitude
    return AffectVector(representation=rep.id, values=values)

In [5]:
# Function to provide an input source that generates a sequence of utterances
# For evaluating the ASA pipeline

import asyncio
from collections.abc import AsyncIterator, Sequence


class UtterancesFromDF:
    """Replays a labelled frame as Utterances carrying their intended affect."""

    def __init__(self, source_df: pd.DataFrame, representation: AffectRepresentation) -> None:
        self._source_df = source_df
        self._rep = representation

    def _intended(self) -> AffectVector:
        test = gen_affect_vector(rep=self._rep, happiness=1.0, surprise=1.0)

        return test

    async def events(self) -> AsyncIterator[Utterance]:

        for _, row in self._source_df.iterrows():
            text = "Nonsense happy text"
            source = "Test text"

            intended = self._intended()
            utterance = Utterance(text=text, source=source, intended=intended)
        
            yield utterance
            await asyncio.sleep(1.0)



In [6]:
# Establish the source, decoder and a simple EKMAN6 keyword decoder

source = UtterancesFromDF(source_df=simple_df, representation=EKMAN6)
decoder = KeywordDecoder(representation=EKMAN6, table=EKMAN6_KEYWORDS)

# Affectmodel is not implemented and just reports a stub
model = AffectModel()

# Observers is not implemented so just produce a debug log
observers = Observers()
observers.register(log_event)

# Kick-off the agent pipeline
await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)

DEBUG: asa.observer.debug.log_event.line_53 - utterance/1  15:56:03.280  753114584183  'Nonsense happy text'
DEBUG: asa.observer.debug.log_event.line_65 - utterance/1  Utterance(text='Nonsense happy text', source='Test text', intended=AffectVector(representation='ekman6/1', values={<SixEmotions.ANGER: 'anger'>: 0.0, <SixEmotions.DISGUST: 'disgust'>: 0.0, <SixEmotions.FEAR: 'fear'>: 0.0, <SixEmotions.HAPPINESS: 'happiness'>: 1.0, <SixEmotions.SADNESS: 'sadness'>: 0.0, <SixEmotions.SURPRISE: 'surprise'>: 1.0}), id='753114584183', at=datetime.datetime(2026, 8, 7, 15, 56, 3, 280518, tzinfo=datetime.timezone.utc), schema='utterance/1')
DEBUG: asa.observer.debug.log_event.line_56 - evidence/1   15:56:03.280  753114584183  other decoder:rule     {'happiness': 0.7}
DEBUG: asa.observer.debug.log_event.line_65 - evidence/1   AffectEvidence(target=<Target.OTHER: 'other'>, affect=AffectVector(representation='ekman6/1', values={<SixEmotions.ANGER: 'anger'>: 0.0, <SixEmotions.DISGUST: 'disgust'>: 0.